# LSTM vs Transformer for Symbolic Music Modeling — Results Analysis

This notebook compares the four model/tokenizer combinations:
- **A**: LSTM + Event-based
- **B**: Transformer + Event-based
- **C**: LSTM + REMI+
- **D**: Transformer + REMI+

Fill in the `results` dict below with numbers from `evaluate.py` output.

In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 11

## 1. Fill in evaluation results

Run `evaluate.py` for each combination and paste the numbers here.

In [ ]:
# Fill these in after running evaluate.py for each combination
results = {
    "A: LSTM + Event": {
        "ce_loss": None,
        "perplexity": None,
        "top1_acc": None,
        "top5_acc": None,
        "bits_per_sec": None,
        "dsr": None,
        # Per-position CE loss
        "pos_ce": {"0-127": None, "128-255": None, "256-383": None, "384-511": None},
        "n_params": None,
    },
    "B: Transformer + Event": {
        "ce_loss": None,
        "perplexity": None,
        "top1_acc": None,
        "top5_acc": None,
        "bits_per_sec": None,
        "dsr": None,
        "pos_ce": {"0-127": None, "128-255": None, "256-383": None, "384-511": None},
        "n_params": None,
    },
    "C: LSTM + REMI+": {
        "ce_loss": None,
        "perplexity": None,
        "top1_acc": None,
        "top5_acc": None,
        "bits_per_sec": None,
        "dsr": None,
        "pos_ce": {"0-127": None, "128-255": None, "256-383": None, "384-511": None},
        "n_params": None,
    },
    "D: Transformer + REMI+": {
        "ce_loss": None,
        "perplexity": None,
        "top1_acc": None,
        "top5_acc": None,
        "bits_per_sec": None,
        "dsr": None,
        "pos_ce": {"0-127": None, "128-255": None, "256-383": None, "384-511": None},
        "n_params": None,
    },
}

## 2. Summary table

In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        "Combination": name,
        "Params": r["n_params"],
        "CE Loss": r["ce_loss"],
        "Perplexity": r["perplexity"],
        "Top-1 Acc": r["top1_acc"],
        "Top-5 Acc": r["top5_acc"],
        "Bits/sec": r["bits_per_sec"],
        "DSR": r["dsr"],
    })

df = pd.DataFrame(rows).set_index("Combination")
df.style.format({
    "CE Loss": "{:.4f}",
    "Perplexity": "{:.2f}",
    "Top-1 Acc": "{:.2%}",
    "Top-5 Acc": "{:.2%}",
    "Bits/sec": "{:.4f}",
    "DSR": "{:.2%}",
})

## 3. Cross-Entropy Loss: LSTM vs Transformer (within each tokenizer)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Event tokenizer: A vs B
event_names = ["A: LSTM + Event", "B: Transformer + Event"]
event_ce = [results[n]["ce_loss"] for n in event_names]
axes[0].bar(event_names, event_ce, color=["steelblue", "darkorange"])
axes[0].set_title("Event Tokenizer: CE Loss")
axes[0].set_ylabel("Cross-Entropy")
for i, v in enumerate(event_ce):
    if v is not None:
        axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center")

# REMI+ tokenizer: C vs D
remi_names = ["C: LSTM + REMI+", "D: Transformer + REMI+"]
remi_ce = [results[n]["ce_loss"] for n in remi_names]
axes[1].bar(remi_names, remi_ce, color=["steelblue", "darkorange"])
axes[1].set_title("REMI+ Tokenizer: CE Loss")
axes[1].set_ylabel("Cross-Entropy")
for i, v in enumerate(remi_ce):
    if v is not None:
        axes[1].text(i, v + 0.01, f"{v:.4f}", ha="center")

plt.tight_layout()
plt.savefig("ce_loss_comparison.png", bbox_inches="tight")
plt.show()

## 4. Per-position loss — does the Transformer's advantage grow at later positions?

In [ ]:
buckets = ["0-127", "128-255", "256-383", "384-511"]
x = np.arange(len(buckets))
width = 0.2

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, (lstm_key, tf_key, title) in zip(
    axes,
    [
        ("A: LSTM + Event", "B: Transformer + Event", "Event Tokenizer"),
        ("C: LSTM + REMI+", "D: Transformer + REMI+", "REMI+ Tokenizer"),
    ],
):
    lstm_vals = [results[lstm_key]["pos_ce"].get(b) for b in buckets]
    tf_vals = [results[tf_key]["pos_ce"].get(b) for b in buckets]

    ax.bar(x - width/2, lstm_vals, width, label="LSTM", color="steelblue")
    ax.bar(x + width/2, tf_vals, width, label="Transformer", color="darkorange")
    ax.set_title(f"{title}: Per-Position CE Loss")
    ax.set_ylabel("Cross-Entropy")
    ax.set_xticks(x)
    ax.set_xticklabels([f"pos {b}" for b in buckets])
    ax.legend()

plt.tight_layout()
plt.savefig("per_position_loss.png", bbox_inches="tight")
plt.show()

## 5. Bits/second — cross-tokenizer comparison

In [ ]:
names = list(results.keys())
bps = [results[n]["bits_per_sec"] for n in names]
colors = ["steelblue", "darkorange", "steelblue", "darkorange"]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(names, bps, color=colors)
ax.set_title("Bits per Second of Musical Time (lower = better)")
ax.set_ylabel("Bits/sec")
for bar, v in zip(bars, bps):
    if v is not None:
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f"{v:.3f}", ha="center", fontsize=9)

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="steelblue", label="LSTM"),
    Patch(facecolor="darkorange", label="Transformer"),
]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.savefig("bits_per_second.png", bbox_inches="tight")
plt.show()

## 6. Top-5 Accuracy

In [ ]:
top5 = [results[n]["top5_acc"] for n in names]
top1 = [results[n]["top1_acc"] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(x - width/2, top1, width, label="Top-1", color="cornflowerblue")
ax.bar(x + width/2, top5, width, label="Top-5", color="coral")
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10)
ax.set_ylabel("Accuracy")
ax.set_title("Top-1 and Top-5 Accuracy")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f"{y:.0%}"))
ax.legend()
plt.tight_layout()
plt.savefig("top_k_accuracy.png", bbox_inches="tight")
plt.show()

## 7. Hypothesis check

The hypothesis states:
> The Transformer achieves lower Perplexity and higher Top-5 Accuracy than the LSTM (within the same tokenizer), and **its advantage grows at later token positions**.

In [ ]:
def check_hypothesis(lstm_key, tf_key, tokenizer_name):
    l = results[lstm_key]
    t = results[tf_key]
    print(f"\n--- {tokenizer_name} ---")

    if None in (l["perplexity"], t["perplexity"]):
        print("  (results not filled in yet)")
        return

    ppl_drop = l["perplexity"] - t["perplexity"]
    print(f"  Perplexity: LSTM={l['perplexity']:.2f}  Transformer={t['perplexity']:.2f}  Δ={ppl_drop:+.2f}")
    print(f"  Transformer wins PPL: {ppl_drop > 0}")

    top5_gain = t["top5_acc"] - l["top5_acc"]
    print(f"  Top-5: LSTM={l['top5_acc']:.2%}  Transformer={t['top5_acc']:.2%}  Δ={top5_gain:+.2%}")
    print(f"  Transformer wins Top-5: {top5_gain > 0}")

    # Per-position: does the gap grow?
    buckets = ["0-127", "128-255", "256-383", "384-511"]
    gaps = []
    for b in buckets:
        lv = l["pos_ce"].get(b)
        tv = t["pos_ce"].get(b)
        if lv is not None and tv is not None:
            gaps.append((b, lv - tv))

    if gaps:
        print("  Per-position CE gap (LSTM - Transformer, higher = Transformer better):")
        for b, g in gaps:
            print(f"    pos {b}: {g:+.4f}")
        gap_values = [g for _, g in gaps]
        growing = all(gap_values[i] <= gap_values[i+1] for i in range(len(gap_values)-1))
        print(f"  Gap grows monotonically: {growing}")

check_hypothesis("A: LSTM + Event", "B: Transformer + Event", "Event tokenizer")
check_hypothesis("C: LSTM + REMI+", "D: Transformer + REMI+", "REMI+ tokenizer")